In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
def process_earthquake_catalog(input_filepath, output_filepath):
    df = pd.read_csv(input_filepath, sep="\t")

    # Clean column names: remove leading/trailing spaces and make them lowercase
    df.columns = df.columns.str.strip().str.lower()

    # Drop rows with invalid dates (often placeholders)
    # The 'date' column in the sample has '????/??/??' which we remove.
    initial_rows = len(df)
    df = df[~df["date"].str.contains("\?")]
    print(f"Removed {initial_rows - len(df)} rows with invalid dates.")

    # Convert relevant columns to numeric, coercing errors to NaN
    # This will handle any non-numeric entries in these columns gracefully.
    cols_to_numeric = ["lat", "long", "mi", "mb", "ms", "mw"]
    for col in cols_to_numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    print("Converted location and magnitude columns to numeric types.")

    # Magnitude Unification ---
    # We will create a single magnitude column 'Mw_unified'.
    # The priority is mw > ms > mb > mi.
    # We use combine_first which takes values from the first series, and fills
    # any missing (NaN) values with values from the second series, and so on.
    df["mw_unified"] = (
        df["mw"].combine_first(df["ms"]).combine_first(df["mb"]).combine_first(df["mi"])
    )
    print("Created 'mw_unified' column with priority: mw > ms > mb > mi.")

    # Drop any rows where we couldn't determine a unified magnitude
    initial_rows = len(df)
    df.dropna(subset=["mw_unified"], inplace=True)
    print(f"Removed {initial_rows - len(df)} events with no magnitude information.")

    # Save the processed data ---
    try:
        df.to_csv(output_filepath, index=False)
        print(f"\nSuccessfully saved cleaned data to: {output_filepath}")
    except Exception as e:
        print(f"Error saving file: {e}")
        return None

    return df

<>:10: SyntaxWarning: invalid escape sequence '\?'
<>:10: SyntaxWarning: invalid escape sequence '\?'
/tmp/ipykernel_188733/441788056.py:10: SyntaxWarning: invalid escape sequence '\?'
  df = df[~df["date"].str.contains("\?")]


In [4]:
input_file = "../assets/data/files/historical_Eq.txt"
output_file = "../assets/data/cleaned_historical_Eq.csv"

cleaned_df = process_earthquake_catalog(input_file, output_file)

if cleaned_df is not None:
    print("First 5 rows of the new catalog:")
    print(cleaned_df.head())
    print("\nData overview:")
    cleaned_df.info()

Removed 1 rows with invalid dates.
Converted location and magnitude columns to numeric types.
Created 'mw_unified' column with priority: mw > ms > mb > mi.
Removed 10 events with no magnitude information.

Successfully saved cleaned data to: ../assets/data/cleaned_historical_Eq.csv
First 5 rows of the new catalog:
         date origine time   lat  long   mi   mb  ms   mw   fd  mw_unified
0  1930/10/02     15:33:12  35.8  52.1  NaN  5.0 NaN  5.1  NaN         5.1
1  1930/10/07     20:53:06  35.8  52.1  NaN  5.0 NaN  5.1  NaN         5.1
2  1945/05/11     20:17:28  34.8  52.1  4.7  NaN NaN  4.6  NaN         4.6
3  1951/04/22     06:32:41  34.8  52.1  5.0  NaN NaN  5.0  NaN         5.0
4  1954/09/02     22:47:00  35.3  52.0  4.5  NaN NaN  4.5  NaN         4.5

Data overview:
<class 'pandas.core.frame.DataFrame'>
Index: 92 entries, 0 to 102
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          92 non-null   

In [9]:
import pandas as pd
import re
import os

def process_stress_file_final(input_filepath, output_directory):
    """
    Parses the specific format of the provided stress.txt file.

    This function is tailored to the file structure where the first four
    datasets are within 'psxy'/'END' blocks, and the final two GPS datasets
    are listed under filename-like headers.

    Args:
        input_filepath (str): The path to the raw stress.txt file.
        output_directory (str): The directory where the cleaned CSVs will be saved.
    """
    print(f"--- Starting final, targeted processing for {input_filepath} ---")

    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"Created output directory: {output_directory}")

    try:
        with open(input_filepath, 'r') as f:
            lines = f.readlines()
        print("Successfully read the stress.txt file.")
    except FileNotFoundError:
        print(f"Error: The file {input_filepath} was not found.")
        return

    data_sections = {
        'stress_RF': [], 'stress_NF': [], 'stress_SS': [],
        'anisotropy_sadidkhuyi2010': [], 'gps_strain_rayisi2016': [],
        'gps_strain_khorrami1390': []
    }
    current_section = None
    data_pattern = re.compile(r'^\s*([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+).*')

    print("\n--- Parsing file with final, targeted logic ---")
    for line in lines:
        stripped_line = line.strip()
        lower_line = line.lower()

        # --- Check for headers to switch sections ---
        # This part handles the first 4 datasets
        if stripped_line.startswith('#'):
            new_section = None
            if 'stress' in lower_line and 'rf' in lower_line:
                new_section = 'stress_RF'
            elif 'stress' in lower_line and 'nf' in lower_line:
                new_section = 'stress_NF'
            elif 'stress' in lower_line and 'ss' in lower_line:
                new_section = 'stress_SS'
            elif 'anisotropy' in lower_line:
                new_section = 'anisotropy_sadidkhuyi2010'

            if new_section:
                current_section = new_section
            continue

        # This part handles the GPS datasets by their specific headers
        if 'gps_strain__rayisi2016.dat' in lower_line:
            current_section = 'gps_strain_rayisi2016'
            continue

        if 'khorrami__1390.dat' in lower_line:
            current_section = 'gps_strain_khorrami1390'
            continue

        # Stop collecting data if an 'END' line is hit for the first 4 datasets
        if stripped_line == 'END':
            current_section = None
            continue

        # --- If we are in an active section, try to parse data ---
        if current_section:
            match = data_pattern.match(line)
            if match:
                lon, lat, val = map(float, match.groups())
                data_sections[current_section].append([lon, lat, val])

    print("Finished parsing. Data points found for each section:")
    for section, data in data_sections.items():
        if data:
            print(f"- {section}: {len(data)} data points")

    print("\n--- Saving extracted data to CSV files ---")
    for section_name, data_list in data_sections.items():
        if data_list:
            df = pd.DataFrame(data_list, columns=['longitude', 'latitude', 'azimuth_value'])
            output_filepath = os.path.join(output_directory, f"{section_name}.csv")
            df.to_csv(output_filepath, index=False)
            print(f"Saved {output_filepath}")

    print("\n--- Processing Complete ---")



In [10]:
input_file = "../assets/data/files/stress.txt"
output_dir = "../assets/data/kinematic_data"

if os.path.exists(output_dir):
    import shutil

    shutil.rmtree(output_dir)

process_stress_file_final(input_file, output_dir)


--- Starting final, targeted processing for ../assets/data/files/stress.txt ---
Created output directory: ../assets/data/kinematic_data
Successfully read the stress.txt file.

--- Parsing file with final, targeted logic ---
Finished parsing. Data points found for each section:
- stress_RF: 4 data points
- stress_NF: 1 data points
- stress_SS: 1 data points
- anisotropy_sadidkhuyi2010: 9 data points
- gps_strain_rayisi2016: 196 data points
- gps_strain_khorrami1390: 11 data points

--- Saving extracted data to CSV files ---
Saved ../assets/data/kinematic_data/stress_RF.csv
Saved ../assets/data/kinematic_data/stress_NF.csv
Saved ../assets/data/kinematic_data/stress_SS.csv
Saved ../assets/data/kinematic_data/anisotropy_sadidkhuyi2010.csv
Saved ../assets/data/kinematic_data/gps_strain_rayisi2016.csv
Saved ../assets/data/kinematic_data/gps_strain_khorrami1390.csv

--- Processing Complete ---


In [11]:
import pandas as pd
import os


def process_vs_file(input_filepath, output_filepath):
    """
    Reads the raw Vs.txt data, skips headers, assigns column names,
    and saves the cleaned data to a new CSV file.

    Args:
        input_filepath (str): The path to the raw Vs.txt file.
        output_filepath (str): The path where the cleaned CSV will be saved.
    """
    print(f"--- Starting processing for {input_filepath} ---")

    try:
        # Use pandas to read the space-separated file.
        # This is highly efficient for large files.
        # skiprows=2 tells it to ignore the first two header lines.
        # sep='\s+' treats one or more spaces as the separator.
        vs_df = pd.read_csv(
            input_filepath,
            sep="\s+",
            skiprows=2,
            header=None,
            names=["longitude", "latitude", "depth", "vs"],
        )
        print(f"Successfully loaded {len(vs_df)} rows from the Vs model.")

    except FileNotFoundError:
        print(f"Error: The file {input_filepath} was not found.")
        return
    except Exception as e:
        print(f"An error occurred: {e}")
        return

    # Display the first 5 rows to confirm it's loaded correctly
    print("\nFirst 5 rows of the loaded Vs data:")
    print(vs_df.head())

    # Display a summary of the data
    print("\nData overview:")
    vs_df.info()

    # Save the cleaned data to a new CSV file for easy access later
    vs_df.to_csv(output_filepath, index=False)
    print(f"\nSuccessfully saved cleaned Vs data to: {output_filepath}")
    print("\n--- Vs data processing complete ---")


# --- Main execution block ---
if __name__ == "__main__":
    # Define the input file and the desired output file
    input_file = "../assets/data/files/Vs.txt"
    output_file = "../assets/data/cleaned_vs_model.csv"

    # Run the processing function
    process_vs_file(input_file, output_file)


<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_203854/4019766192.py:23: SyntaxWarning: invalid escape sequence '\s'
  sep="\s+",


--- Starting processing for ../assets/data/files/Vs.txt ---
Successfully loaded 87750 rows from the Vs model.

First 5 rows of the loaded Vs data:
   longitude   latitude  depth      vs
0  50.042986  35.979592   0.25  2.8755
1  50.144796  35.979592   0.25  2.8645
2  50.246606  35.979592   0.25  2.8514
3  50.348416  35.979592   0.25  2.8412
4  50.450226  35.979592   0.25  2.8369

Data overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87750 entries, 0 to 87749
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   longitude  87750 non-null  float64
 1   latitude   87750 non-null  float64
 2   depth      87750 non-null  float64
 3   vs         87750 non-null  float64
dtypes: float64(4)
memory usage: 2.7 MB

Successfully saved cleaned Vs data to: ../assets/data/cleaned_vs_model.csv

--- Vs data processing complete ---


In [13]:
import pandas as pd
import numpy as np
import os


def create_study_grid(output_filepath, grid_spacing=0.1):
    """
    Creates a uniform grid of coordinates covering the full extent of the study area.

    The function determines the study area boundaries by loading all previously
    cleaned datasets (earthquakes and Vs model) and finding the overall minimum
    and maximum latitude and longitude. It then generates a grid of points with
    the specified spacing and saves it to a CSV file.

    Args:
        output_filepath (str): The path to save the generated grid CSV file.
        grid_spacing (float): The resolution of the grid in degrees (e.g., 0.1).
    """
    print("--- Starting Step 2: Creating the study grid ---")

    # --- 1. Load all relevant datasets to determine boundaries ---
    try:
        eq_df = pd.read_csv("../assets/data/cleaned_historical_Eq.csv")
        vs_df = pd.read_csv("../assets/data/cleaned_vs_model.csv")
        print("Successfully loaded cleaned earthquake and Vs model data.")
    except FileNotFoundError as e:
        print(f"Error: A required data file was not found. {e}")
        print(
            "Please ensure 'cleaned_earthquake_catalog.csv' and 'cleaned_vs_model.csv' are present."
        )
        return

    # --- 2. Determine the geographic extent of all data ---
    all_lons = pd.concat([eq_df["long"], vs_df["longitude"]])
    all_lats = pd.concat([eq_df["lat"], vs_df["latitude"]])

    min_lon, max_lon = all_lons.min(), all_lons.max()
    min_lat, max_lat = all_lats.min(), all_lats.max()

    # Add a small buffer to the boundaries to ensure full coverage
    buffer = grid_spacing
    grid_min_lon = np.floor(min_lon / grid_spacing) * grid_spacing - buffer
    grid_max_lon = np.ceil(max_lon / grid_spacing) * grid_spacing + buffer
    grid_min_lat = np.floor(min_lat / grid_spacing) * grid_spacing - buffer
    grid_max_lat = np.ceil(max_lat / grid_spacing) * grid_spacing + buffer

    print("\nDetermined grid boundaries:")
    print(f"  Longitude: {grid_min_lon:.2f} to {grid_max_lon:.2f}")
    print(f"  Latitude:  {grid_min_lat:.2f} to {grid_max_lat:.2f}")

    # --- 3. Create the grid coordinates ---
    lon_points = np.arange(grid_min_lon, grid_max_lon, grid_spacing)
    lat_points = np.arange(grid_min_lat, grid_max_lat, grid_spacing)

    # Use meshgrid to create a coordinate matrix for every point
    xx, yy = np.meshgrid(lon_points, lat_points)

    # Flatten the matrices to create columns of coordinates
    grid_df = pd.DataFrame({"longitude": xx.ravel(), "latitude": yy.ravel()})

    print(f"\nGenerated a grid of {len(lon_points)} x {len(lat_points)}.")
    print(f"Total grid cells to process: {len(grid_df)}")

    # --- 4. Save the grid to a file ---
    grid_df.to_csv(output_filepath, index=False)
    print(f"\nSuccessfully saved the study grid to: {output_filepath}")
    print("\n--- Grid creation complete ---")


# --- Main execution block ---
if __name__ == "__main__":
    # Define the output file for our new grid
    output_file = "study_grid.csv"

    # Run the grid creation function
    create_study_grid(output_file, grid_spacing=0.1)


--- Starting Step 2: Creating the study grid ---
Successfully loaded cleaned earthquake and Vs model data.

Determined grid boundaries:
  Longitude: 49.90 to 54.10
  Latitude:  34.40 to 36.80

Generated a grid of 43 x 25.
Total grid cells to process: 1075

Successfully saved the study grid to: study_grid.csv

--- Grid creation complete ---


In [16]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import os


def calculate_structural_features(
    grid_filepath, vs_filepath, output_filepath, search_radius=0.15
):
    """
    Calculates structural features (mean and std dev of Vs) for each point in a study grid.

    This function loads a grid of coordinates and a dense set of shear-wave velocity (Vs)
    data points. It uses a KDTree for efficient spatial searching to find all Vs points
    within a specified radius of each grid point. It then computes the mean and
    standard deviation of the Vs values for these neighbors and appends them as new
    features to the grid.

    Args:
        grid_filepath (str): Path to the study grid CSV file.
        vs_filepath (str): Path to the cleaned Vs model CSV file.
        output_filepath (str): Path to save the grid with new structural features.
        search_radius (float): The search radius in degrees to find neighbors.
    """
    print("--- Starting Step 3: Calculating structural features ---")

    # --- 1. Load the grid and the Vs data ---
    try:
        grid_df = pd.read_csv(grid_filepath)
        vs_df = pd.read_csv(vs_filepath)
        print(f"Loaded {len(grid_df)} grid points and {len(vs_df)} Vs data points.")
    except FileNotFoundError as e:
        print(f"Error: A required data file was not found. {e}")
        return

    # --- 2. Build a KDTree for efficient spatial searching ---
    # We build the tree on the Vs data coordinates to quickly find neighbors.
    vs_coordinates = vs_df[["longitude", "latitude"]].values
    vs_tree = cKDTree(vs_coordinates)
    print(
        f"Built KDTree from Vs data for fast neighbor search (radius = {search_radius}°)."
    )

    # --- 3. Calculate features for each grid point ---
    vs_means = []
    vs_stds = []

    print("\nCalculating features for each grid point...")
    # Iterate through each row (each grid cell) in our study grid
    for index, row in grid_df.iterrows():
        grid_point = [row["longitude"], row["latitude"]]

        # Query the tree to find indices of all Vs points within the search radius
        neighbor_indices = vs_tree.query_ball_point(grid_point, r=search_radius)

        if neighbor_indices:
            # If neighbors are found, get their Vs values
            neighbor_vs = vs_df.loc[neighbor_indices, "vs"]

            # Calculate mean and standard deviation and append to our lists
            vs_means.append(neighbor_vs.mean())
            vs_stds.append(neighbor_vs.std())
        else:
            # If no neighbors are found (unlikely), append NaN
            vs_means.append(np.nan)
            vs_stds.append(np.nan)

        # Print a progress update every 100 points
        if (index + 1) % 100 == 0:
            print(f"  Processed {index + 1}/{len(grid_df)} points...")

    print("...Calculation complete.")

    # --- 4. Add the new features to the grid DataFrame ---
    grid_df["vs_mean"] = vs_means
    grid_df["vs_std"] = vs_stds

    # UPDATED: Replaced inplace=True to align with modern pandas practices
    grid_df["vs_mean"] = grid_df["vs_mean"].fillna(grid_df["vs_mean"].mean())
    grid_df["vs_std"] = grid_df["vs_std"].fillna(0)

    print("\nFirst 5 rows of the grid with new features:")
    print(grid_df.head())

    # --- 5. Save the result to a new file ---
    grid_df.to_csv(output_filepath, index=False)
    print(f"\nSuccessfully saved grid with structural features to: {output_filepath}")
    print("\n--- Structural feature calculation complete ---")


# --- Main execution block ---
if __name__ == "__main__":
    # Define the input and output files
    grid_file = "study_grid.csv"
    vs_file = "../assets/data/cleaned_vs_model.csv"
    output_file = "grid_with_structural_features.csv"

    # Run the feature calculation function
    calculate_structural_features(grid_file, vs_file, output_file)


--- Starting Step 3: Calculating structural features ---
Loaded 1075 grid points and 87750 Vs data points.
Built KDTree from Vs data for fast neighbor search (radius = 0.15°).

Calculating features for each grid point...
  Processed 100/1075 points...
  Processed 200/1075 points...
  Processed 300/1075 points...
  Processed 400/1075 points...
  Processed 500/1075 points...
  Processed 600/1075 points...
  Processed 700/1075 points...
  Processed 800/1075 points...
  Processed 900/1075 points...
  Processed 1000/1075 points...
...Calculation complete.

First 5 rows of the grid with new features:
   longitude  latitude   vs_mean  vs_std
0       49.9      34.4  3.499065     0.0
1       50.0      34.4  3.499065     0.0
2       50.1      34.4  3.499065     0.0
3       50.2      34.4  3.499065     0.0
4       50.3      34.4  3.499065     0.0

Successfully saved grid with structural features to: grid_with_structural_features.csv

--- Structural feature calculation complete ---


In [18]:
import pandas as pd
from scipy.spatial import cKDTree
import os

def calculate_kinematic_features(grid_filepath, kinematic_data_dir, output_filepath):
    """
    Calculates kinematic features by finding the nearest GPS strain measurement
    for each point in the study grid.

    This function loads the grid (which already has structural features),
    combines all available GPS strain data, and then uses a KDTree to efficiently
    find the single nearest GPS station for each grid point. It assigns the
    azimuth value from that nearest station as a new feature.

    Args:
        grid_filepath (str): Path to the grid CSV file (with structural features).
        kinematic_data_dir (str): Path to the directory containing kinematic CSVs.
        output_filepath (str): Path to save the grid with new kinematic features.
    """
    print("--- Starting Step 4: Calculating kinematic features ---")

    # --- 1. Load the grid data ---
    try:
        grid_df = pd.read_csv(grid_filepath)
        print(f"Loaded {len(grid_df)} grid points from '{grid_filepath}'.")
    except FileNotFoundError:
        print(f"Error: The grid file '{grid_filepath}' was not found.")
        return

    # --- 2. Load and combine all GPS strain data ---
    gps_files = [
        'gps_strain_rayisi2016.csv',
        'gps_strain_khorrami1390.csv'
    ]
    all_gps_data = []
    print("\nLoading GPS strain data...")
    for f in gps_files:
        filepath = os.path.join(kinematic_data_dir, f)
        try:
            gps_df = pd.read_csv(filepath)
            all_gps_data.append(gps_df)
            print(f"- Loaded {len(gps_df)} points from '{f}'")
        except FileNotFoundError:
            print(f"- Warning: GPS file not found at '{filepath}'. Skipping.")

    if not all_gps_data:
        print("Error: No GPS data could be loaded. Cannot proceed.")
        return

    combined_gps_df = pd.concat(all_gps_data, ignore_index=True)
    print(f"Total of {len(combined_gps_df)} GPS data points available.")

    # --- 3. Build KDTree from GPS data for nearest neighbor search ---
    gps_coordinates = combined_gps_df[['longitude', 'latitude']].values
    gps_tree = cKDTree(gps_coordinates)
    print("Built KDTree from GPS data for fast nearest neighbor search.")

    # --- 4. Find the nearest GPS measurement for each grid point ---
    grid_coordinates = grid_df[['longitude', 'latitude']].values

    # query the tree to find the distance and index of the single nearest neighbor (k=1)
    distances, indices = gps_tree.query(grid_coordinates, k=1)

    # Use the indices to get the azimuth values from the combined GPS dataframe
    nearest_azimuths = combined_gps_df.loc[indices, 'azimuth_value'].values

    # --- 5. Add the new feature to the grid DataFrame ---
    grid_df['strain_azimuth'] = nearest_azimuths
    print("\nAssigned nearest strain azimuth to each grid point.")

    print("\nFirst 5 rows of the grid with new kinematic feature:")
    print(grid_df.head())

    # --- 6. Save the result to a new file ---
    grid_df.to_csv(output_filepath, index=False)
    print(f"\nSuccessfully saved grid with kinematic features to: {output_filepath}")
    print("\n--- Kinematic feature calculation complete ---")


# --- Main execution block ---
if __name__ == '__main__':
    # Define input and output files/directories
    grid_file = 'grid_with_structural_features.csv'
    kinematic_dir = '../assets/data/kinematic_data'
    output_file = 'grid_with_kinematic_features.csv'

    # Run the feature calculation function
    calculate_kinematic_features(grid_file, kinematic_dir, output_file)


--- Starting Step 4: Calculating kinematic features ---
Loaded 1075 grid points from 'grid_with_structural_features.csv'.

Loading GPS strain data...
- Loaded 196 points from 'gps_strain_rayisi2016.csv'
- Loaded 11 points from 'gps_strain_khorrami1390.csv'
Total of 207 GPS data points available.
Built KDTree from GPS data for fast nearest neighbor search.

Assigned nearest strain azimuth to each grid point.

First 5 rows of the grid with new kinematic feature:
   longitude  latitude   vs_mean  vs_std  strain_azimuth
0       49.9      34.4  3.499065     0.0          107.30
1       50.0      34.4  3.499065     0.0          107.30
2       50.1      34.4  3.499065     0.0          107.30
3       50.2      34.4  3.499065     0.0          107.30
4       50.3      34.4  3.499065     0.0           96.64

Successfully saved grid with kinematic features to: grid_with_kinematic_features.csv

--- Kinematic feature calculation complete ---


In [19]:
import pandas as pd
from scipy.spatial import cKDTree
import os

def create_target_variable(grid_filepath, eq_filepath, output_filepath, search_radius=0.15):
    """
    Creates the final machine learning dataset by adding the target variable.

    This function loads the grid with all calculated features and the cleaned
    earthquake catalog. It then creates a binary target variable ('earthquake_occurred')
    for each grid cell, labeling it '1' if any historical earthquake occurred
    within a specified radius, and '0' otherwise.

    Args:
        grid_filepath (str): Path to the grid CSV with structural/kinematic features.
        eq_filepath (str): Path to the cleaned earthquake catalog CSV.
        output_filepath (str): Path to save the final, complete feature matrix.
        search_radius (float): The radius in degrees to search for earthquakes.
    """
    print("--- Starting Step 5: Creating the target variable ---")

    # --- 1. Load the grid with features and the earthquake data ---
    try:
        grid_df = pd.read_csv(grid_filepath)
        eq_df = pd.read_csv(eq_filepath)
        print(f"Loaded {len(grid_df)} grid points and {len(eq_df)} earthquakes.")
    except FileNotFoundError as e:
        print(f"Error: A required data file was not found. {e}")
        return

    # --- 2. Build KDTree from earthquake locations ---
    eq_coordinates = eq_df[['long', 'lat']].values
    eq_tree = cKDTree(eq_coordinates)
    print(f"Built KDTree from earthquake data for proximity search (radius = {search_radius}°).")

    # --- 3. Determine if an earthquake occurred near each grid point ---
    target_variable = []
    print("\nLabeling each grid point (0=non-hazardous, 1=hazardous)...")

    for index, row in grid_df.iterrows():
        grid_point = [row['longitude'], row['latitude']]

        # Query the tree to find if ANY earthquake is within the radius.
        # We only need to know if the list of neighbors is empty or not.
        neighbor_indices = eq_tree.query_ball_point(grid_point, r=search_radius)

        if neighbor_indices:
            # If the list is not empty, an earthquake occurred nearby.
            target_variable.append(1)
        else:
            # If the list is empty, no earthquake was nearby.
            target_variable.append(0)

        if (index + 1) % 100 == 0:
            print(f"  Processed {index + 1}/{len(grid_df)} points...")

    print("...Labeling complete.")

    # --- 4. Add the target variable to the DataFrame ---
    grid_df['earthquake_occurred'] = target_variable

    # --- 5. Final check and save ---
    print(f"\nTotal hazardous cells (1): {grid_df['earthquake_occurred'].sum()}")
    print(f"Total non-hazardous cells (0): {len(grid_df) - grid_df['earthquake_occurred'].sum()}")

    print("\nFirst 5 rows of the final dataset:")
    print(grid_df.head())

    grid_df.to_csv(output_filepath, index=False)
    print(f"\nSuccessfully saved the final feature matrix to: {output_filepath}")
    print("\n--- Feature engineering complete! ---")
    print("The dataset is now ready for machine learning.")

# --- Main execution block ---
if __name__ == '__main__':
    # Define input and output files
    grid_file = 'grid_with_kinematic_features.csv'
    eq_file = '../assets/data/cleaned_historical_Eq.csv'
    output_file = 'final_feature_matrix.csv'

    # Run the function to create the final dataset
    create_target_variable(grid_file, eq_file, output_file)


--- Starting Step 5: Creating the target variable ---
Loaded 1075 grid points and 92 earthquakes.
Built KDTree from earthquake data for proximity search (radius = 0.15°).

Labeling each grid point (0=non-hazardous, 1=hazardous)...
  Processed 100/1075 points...
  Processed 200/1075 points...
  Processed 300/1075 points...
  Processed 400/1075 points...
  Processed 500/1075 points...
  Processed 600/1075 points...
  Processed 700/1075 points...
  Processed 800/1075 points...
  Processed 900/1075 points...
  Processed 1000/1075 points...
...Labeling complete.

Total hazardous cells (1): 240
Total non-hazardous cells (0): 835

First 5 rows of the final dataset:
   longitude  latitude   vs_mean  vs_std  strain_azimuth  earthquake_occurred
0       49.9      34.4  3.499065     0.0          107.30                    0
1       50.0      34.4  3.499065     0.0          107.30                    0
2       50.1      34.4  3.499065     0.0          107.30                    0
3       50.2      34.